 ## Tahmine Dayalı Stok Planlama

Bu notebook, 14 günlük talep tahminlerini başlangıç stoklarıyla
karşılaştırarak stok tükenme durumunu ve sipariş ihtiyacını hesaplar.

Günlük stok projeksiyonu, yeni stok girişi olmadığı varsayımına dayanır.
Sipariş önerileri 3 günlük tedarik süresi ve güvenlik stoğu kullanılarak oluşturulur.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path 
from scipy.stats import norm

 ## 2. Talep Tahminlerini Yükleme

Önceden üretilmiş günlük talep tahminleri dosyadan okunur.
Mağaza ve ürün sütunlarının adları, stok tablosuyla eşleşecek şekilde düzenlenir.

Her satır bir tarih, mağaza ve ürün için tahmini talebi gösterir.
`head()` yalnızca ilk 5 satırı görüntüler.

In [2]:
# DUZELTME: eski path "data\\processed\\forecast\\future_forecast_14d.parquet"
# hem Windows'a ozgu ters slash kullaniyordu (Linux/Mac'te calismaz ve
# "\f" gibi kombinasyonlar Python string'inde kacis karakteri olarak
# yorumlanabilir) hem de "../" onekini unutmustu: 04_future_forecast_fixed
# notebook'u ciktiyi "../data/processed/forecast/..." konumuna (proje
# kok dizinindeki data/ klasorune) kaydediyor, bu da calisma sirasinda
# gorulen FileNotFoundError'in sebebiydi.
forecast_14d = pd.read_parquet(
    Path("../data/processed/forecast/future_forecast_14d.parquet")
).rename(columns={
    "Store_ID": "Store ID",
    "Product_ID": "Product ID"
})

forecast_14d.head()

,Date,Store ID,Product ID,Forecast Demand
0,2023-11-01,S001,P0001,114.017840
1,2023-11-01,S001,P0002,162.974963
2,2023-11-01,S001,P0003,182.341952
3,2023-11-01,S001,P0004,139.060325
4,2023-11-01,S001,P0005,230.250066


 ## 3. Tahmin Kapsamını Kontrol Etme

Toplam kayıt sayısı, farklı tarih sayısı ve günlük kayıt sayıları incelenir.
Amaç, tahminlerin beklenen 14 günlük dönemi kapsadığını kontrol etmektir.

In [3]:
print("Toplam satır:", len(forecast_14d))
print("Farklı gün sayısı:", forecast_14d["Date"].nunique())
print(forecast_14d.groupby("Date").size())

Toplam satır: 1400
Farklı gün sayısı: 14
Date
2023-11-01    100
2023-11-02    100
2023-11-03    100
2023-11-04    100
2023-11-05    100
2023-11-06    100
2023-11-07    100
2023-11-08    100
2023-11-09    100
2023-11-10    100
2023-11-11    100
2023-11-12    100
2023-11-13    100
2023-11-14    100
dtype: int64


 ## 4. Örnek Ürünün Günlük Tahminleri

S001 mağazasındaki P0001 ürününün tahminleri tarihe göre gösterilir.
Bu görünüm, aynı ürünün talebinin 14 gün boyunca nasıl değiştiğini incelemeyi sağlar.
sayıları yuvarlayarak okumak lazım

In [4]:
display(
    forecast_14d[
        (forecast_14d["Store ID"] == "S001") &
        (forecast_14d["Product ID"] == "P0001")
    ].sort_values("Date")
)

,Date,Store ID,Product ID,Forecast Demand
0,2023-11-01,S001,P0001,114.01784
100,2023-11-02,S001,P0001,114.01784
200,2023-11-03,S001,P0001,114.01784
300,2023-11-04,S001,P0001,114.01784
400,2023-11-05,S001,P0001,114.01784
500,2023-11-06,S001,P0001,114.01784
600,2023-11-07,S001,P0001,114.01784
700,2023-11-08,S001,P0001,114.01784
800,2023-11-09,S001,P0001,114.01784
900,2023-11-10,S001,P0001,114.01784


 ## 5. Geçmiş Verileri Hazırlama

doğrulama ve test dosyaları tek tabloda birleştirilir.
Yalnızca tahmin başlangıcından önceki kayıtlar kullanılır.

Bu veriler başlangıç stoğunu ve geçmiş satış değişkenliğini hesaplamak için gerekli

In [5]:
# DUZELTME: yanlislikla eklenmis "notebooks/" oneki kaldirildi ve
# 04_future_forecast_fixed.ipynb ile ayni "../data/processed/main/..."
# konumu kullanildi (her iki notebook da notebooks/ klasorunden
# calistiriliyor, bu yuzden proje kokune "../" ile erisilmesi gerekiyor).
df_history = pd.concat([
    pd.read_parquet(Path("../data/processed/main/train.parquet")),
    pd.read_parquet(Path("../data/processed/main/validation.parquet")),
    pd.read_parquet(Path("../data/processed/main/test.parquet")),
], ignore_index=True)

df_history.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,...,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2022-01-01,S001,P0001,231,127,55,33.50,20,0,2022,...,0,1,0,0,1,0,0,0,0,1
1,2022-01-02,S001,P0001,116,81,104,27.95,10,0,2022,...,0,0,0,1,0,0,0,0,0,1
2,2022-01-03,S001,P0001,154,5,189,62.70,20,0,2022,...,0,0,0,1,1,0,0,0,0,1
3,2022-01-04,S001,P0001,85,58,193,77.88,15,1,2022,...,0,0,1,0,0,0,0,0,0,1
4,2022-01-05,S001,P0001,238,147,37,28.46,20,1,2022,...,0,0,1,0,0,0,1,0,0,1


## 6. Başlangıç Stoğunu Belirleme

Her mağaza–ürün için son bilinen stok değeri seçilir.
Bu değer, tahmin dönemi boyunca yapılacak stok hesabının başlangıç noktasıdır.

In [6]:
latest_inventory = (
    df_history
    .sort_values("Date")
    .groupby(["Store ID", "Product ID"])["Inventory Level"]
    .last()
    .reset_index()
)

latest_inventory.head()

,Store ID,Product ID,Inventory Level
0,S001,P0001,223
1,S001,P0002,217
2,S001,P0003,69
3,S001,P0004,338
4,S001,P0005,471


## 7. Tahmin ve Stok Verilerini Birleştirme

Her günlük tahmin kaydına ilgili mağaza–ürünün başlangıç stoğu eklenir.
Eşleştirme mağaza ve ürün kimlikleri üzerinden yapılır.

In [7]:
inventory_forecast = forecast_14d.merge(
    latest_inventory,
    on=["Store ID", "Product ID"],
    how="left"
)

inventory_forecast.head()

,Date,Store ID,Product ID,Forecast Demand,Inventory Level
0,2023-11-01,S001,P0001,114.017840,223
1,2023-11-01,S001,P0002,162.974963,217
2,2023-11-01,S001,P0003,182.341952,69
3,2023-11-01,S001,P0004,139.060325,338
4,2023-11-01,S001,P0005,230.250066,471


## 8. Günlük Stok Projeksiyonu

Günlük tahmini talepler her mağaza–ürün için biriktirilir.

- **Cumulative Forecast:** İlgili güne kadar toplam tahmini talep.
- **Projected Inventory:** Başlangıç stoğundan birikimli talep çıkarıldıktan sonraki bakiye.

Yeni stok girişi hesaba katılmaz.
Negatif bakiye, karşılanamayan talebi ifade eder.

In [8]:
inventory_forecast = inventory_forecast.sort_values(
    ["Store ID", "Product ID", "Date"]
).copy()

inventory_forecast["Cumulative Forecast"] = (
    inventory_forecast
    .groupby(["Store ID", "Product ID"])["Forecast Demand"]
    .cumsum()
)

inventory_forecast["Projected Inventory"] = (
    inventory_forecast["Inventory Level"]
    - inventory_forecast["Cumulative Forecast"]
)

inventory_forecast.head()


,Date,Store ID,Product ID,Forecast Demand,Inventory Level,Cumulative Forecast,Projected Inventory
0,2023-11-01,S001,P0001,114.01784,223,114.017840,108.982160
100,2023-11-02,S001,P0001,114.01784,223,228.035681,-5.035681
200,2023-11-03,S001,P0001,114.01784,223,342.053521,-119.053521
300,2023-11-04,S001,P0001,114.01784,223,456.071362,-233.071362
400,2023-11-05,S001,P0001,114.01784,223,570.089202,-347.089202


## 9. Günlük Stok Tükenme Durumu

Tahmini stok bakiyesi sıfır veya altındaysa HIGH, pozitifse LOW etiketi atanır.
Bu etiket bir olasılık değil, hesaplanan stok bakiyesine dayalı bir işarettir.

In [9]:
inventory_forecast["Stockout Risk"] = np.where(
    inventory_forecast["Projected Inventory"] <= 0,
    "HIGH",
    "LOW")

## 10. Mağaza–Ürün Bazında Dönem Özeti

Günlük kayıtlar her mağaza–ürün için tek satırda özetlenir.

- **Current_Inventory:** Başlangıç stoğu.
- **Forecast_7D:** İlk 7 günlük toplam tahmini talep.
- **Forecast_14D:** 14 günlük toplam tahmini talep.
- **Min_Projected_Inventory:** Dönem içindeki en düşük stok bakiyesi.

In [10]:
risk_summary = (
    inventory_forecast
    .groupby(["Store ID", "Product ID"])
    .agg(
        Current_Inventory=("Inventory Level", "first"),
        Min_Projected_Inventory=("Projected Inventory", "min"),
        Forecast_14D=("Forecast Demand", "sum"),
    )
    .reset_index()
)

# Forecast_7D icin ayri bir hesap: sadece ilk 7 gunun toplami
forecast_7d = (
    inventory_forecast
    .sort_values(["Store ID", "Product ID", "Date"])
    .groupby(["Store ID", "Product ID"])["Forecast Demand"]
    .apply(lambda x: x.iloc[:7].sum())
    .reset_index(name="Forecast_7D")
)

risk_summary = risk_summary.merge(forecast_7d, on=["Store ID", "Product ID"], how="left")

risk_summary.head()

,Store ID,Product ID,Current_Inventory,Min_Projected_Inventory,Forecast_14D,Forecast_7D
0,S001,P0001,223,-1373.249767,1596.249767,798.124883
1,S001,P0002,217,-2064.649484,2281.649484,1140.824742
2,S001,P0003,69,-2483.787330,2552.787330,1276.393665
3,S001,P0004,338,-1611.311837,1949.311837,973.422278
4,S001,P0005,471,-2752.500929,3223.500929,1611.750464


## 11. Dönem İçinde Stok Tükenme Durumu

14 günlük dönem içinde stok bakiyesi sıfıra veya altına düşüyorsa HIGH etiketi verilir.
Bu değerlendirme, yeni stok gelmediği senaryoyu gösterir.

In [11]:
risk_summary["Stockout Risk"] = np.where(
    risk_summary["Min_Projected_Inventory"] <= 0,
    "HIGH",
    "LOW")

## 12. Stokun Tahmini Yetme Süresi

14 günlük toplam talep 14'e bölünerek ortalama günlük talep hesaplanır.
Başlangıç stoğunun bu ortalamaya bölünmesiyle Days_of_Cover elde edilir.

Days_of_Cover, stokun yaklaşık kaç gün yeteceğini gösterir;
günlük talep değiştiği için kesin stok tükenme tarihi değildir.

In [12]:
risk_summary["Avg_Daily_Forecast"] = (risk_summary["Forecast_14D"] / 14)
risk_summary["Days_of_Cover"] = (risk_summary["Current_Inventory"]
    / risk_summary["Avg_Daily_Forecast"].replace(0, np.nan))

In [13]:
risk_summary[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Avg_Daily_Forecast",
        "Days_of_Cover"]].head(20)

,Store ID,Product ID,Current_Inventory,Avg_Daily_Forecast,Days_of_Cover
0,S001,P0001,223,114.017840,1.955834
1,S001,P0002,217,162.974963,1.331493
2,S001,P0003,69,182.341952,0.378410
3,S001,P0004,338,139.236560,2.427523
4,S001,P0005,471,230.250066,2.045602
5,S001,P0006,305,89.262786,3.416877
6,S001,P0007,256,85.408061,2.997375
7,S001,P0008,315,128.820474,2.445263
8,S001,P0009,167,207.843598,0.803489
9,S001,P0010,167,61.391835,2.720231


## 13. Stok Yeterliliğini Sınıflandırma

Ortalama talebe göre stokun yetme süresi sınıflandırılır:

- **HIGH:** 3 günden az.
- **MEDIUM:** En az 3, ancak 7 günden az.
- **LOW:** 7–14 gün.
- **OVERSTOCK:** 14 günden fazla.

Bu sınıflandırma, dönem içinde stok tükenmesini gösteren Stockout Risk
sütunundan farklıdır. OVERSTOCK etiketi, bu çalışmadaki 14 günlük eşiğe dayanır.

In [14]:
def classify_stock_risk(days):

    if pd.isna(days):
        return "NO DEMAND"

    elif days < 3:
        return "HIGH"

    elif days < 7:
        return "MEDIUM"

    elif days <= 14:
        return "LOW"

    else:
        return "OVERSTOCK"

In [15]:
risk_summary["Risk Level"] = (
    risk_summary["Days_of_Cover"]
    .apply(classify_stock_risk))

In [16]:
risk_summary[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Forecast_7D",
        "Forecast_14D",
        "Days_of_Cover",
        "Stockout Risk",
        "Risk Level"]].head(20)

,Store ID,Product ID,Current_Inventory,Forecast_7D,Forecast_14D,Days_of_Cover,Stockout Risk,Risk Level
0,S001,P0001,223,798.124883,1596.249767,1.955834,HIGH,HIGH
1,S001,P0002,217,1140.824742,2281.649484,1.331493,HIGH,HIGH
2,S001,P0003,69,1276.393665,2552.787330,0.378410,HIGH,HIGH
3,S001,P0004,338,973.422278,1949.311837,2.427523,HIGH,HIGH
4,S001,P0005,471,1611.750464,3223.500929,2.045602,HIGH,HIGH
5,S001,P0006,305,624.839502,1249.679003,3.416877,HIGH,MEDIUM
6,S001,P0007,256,597.856430,1195.712860,2.997375,HIGH,HIGH
7,S001,P0008,315,901.743315,1803.486630,2.445263,HIGH,HIGH
8,S001,P0009,167,1454.905189,2909.810379,0.803489,HIGH,HIGH
9,S001,P0010,167,429.742844,859.485688,2.720231,HIGH,HIGH


In [17]:
risk_summary["Risk Level"].value_counts()

HIGH      71
MEDIUM    28
LOW        1
Name: Risk Level, dtype: int64

## 14. Stok Yeterliliği Dağılımı

Her risk sınıfındaki mağaza–ürün kayıtları sayılır.
Aynı ürün farklı mağazalarda ayrı değerlendirilir.

In [18]:
LEAD_TIME_DAYS = 3
SERVICE_LEVEL_Z = 1.65

## 16. Geçmiş Satış İstatistikleri

Her mağaza–ürün için geçmiş satışların ortalaması ve standart sapması hesaplanır.

- **Demand_Mean:** Geçmiş ortalama satış.
- **Demand_Std:** Geçmiş satışların değişkenliği.

Standart sapma güvenlik stoğu hesabında kullanılır.

In [19]:
demand_stats = (
    df_history
    .groupby(["Store ID", "Product ID"])["Units Sold"]
    .agg(
        Demand_Mean="mean",
        Demand_Std="std"
    )
    .reset_index()
)

demand_stats.head()

,Store ID,Product ID,Demand_Mean,Demand_Std
0,S001,P0001,137.306430,108.187859
1,S001,P0002,131.038304,104.313220
2,S001,P0003,141.440492,107.211225
3,S001,P0004,140.904241,113.644117
4,S001,P0005,133.129959,107.368873


In [20]:
risk_summary = risk_summary.merge(
    demand_stats,
    on=["Store ID", "Product ID"],
    how="left"
)

## 17. Güvenlik Stoğu

Talep değişkenliğine karşı  stok hesaplanır:

Safety_Stock = 1,65 × Demand_Std × √Tedarik Süresi

Bu basit yaklaşım, günlük talep değişkenliğinin dönem boyunca benzer
olduğunu ve günler arası bağımlılığın ihmal edilebildiğini varsayar.

In [21]:
risk_summary["Safety_Stock"] = (
    SERVICE_LEVEL_Z
    * risk_summary["Demand_Std"]
    * np.sqrt(LEAD_TIME_DAYS)
)

In [22]:
risk_summary["Safety_Stock"] = (
    risk_summary["Safety_Stock"]
    .fillna(0)
    .clip(lower=0)
)

## 18. Tedarik Süresindeki Talep

İlk 3 günlük tahmin toplanarak sipariş gelene kadar beklenen talep hesaplanır.
Sonuç Lead_Time_Demand sütununda tutulur.

In [23]:
lead_time_forecast = (
    forecast_14d
    .sort_values(["Store ID", "Product ID", "Date"])
    .groupby(["Store ID", "Product ID"])
    ["Forecast Demand"]
    .apply(lambda x: x.iloc[:LEAD_TIME_DAYS].sum())
    .reset_index(name="Lead_Time_Demand")
)

In [24]:
risk_summary = risk_summary.merge(
    lead_time_forecast,
    on=["Store ID", "Product ID"],
    how="left"
)

## 19. Hedef Stok

Tedarik süresindeki tahmini talebe güvenlik stoğu eklenir:

Target_Stock = Lead_Time_Demand + Safety_Stock

Bu hedef, doğrudan 14 günlük toplam talebi değil,
tedarik süresini ve tampon stoğu temel alır.

In [25]:
risk_summary["Target_Stock"] = (
    risk_summary["Lead_Time_Demand"]
    + risk_summary["Safety_Stock"]
)

## 20. Sipariş Önerisi

Hedef stoktan mevcut stok çıkarılarak sipariş ihtiyacı hesaplanır.
Negatif sonuçlar sıfırlanır ve miktarlar tam adede yukarı yuvarlanır.

Mevcut açık siparişler, minimum sipariş miktarı ve koli büyüklüğü
bu hesapta dikkate alınmaz.

In [26]:
risk_summary["Recommended_Order_Qty"] = (
    risk_summary["Target_Stock"]
    - risk_summary["Current_Inventory"]
)

In [27]:
risk_summary["Recommended_Order_Qty"] = (
    risk_summary["Recommended_Order_Qty"]
    .clip(lower=0)
)

In [28]:
risk_summary["Recommended_Order_Qty"] = (
    np.ceil(
        risk_summary["Recommended_Order_Qty"]
    ).astype(int)
)

In [29]:
final_risk_analysis = risk_summary[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Forecast_7D",
        "Forecast_14D",
        "Avg_Daily_Forecast",
        "Days_of_Cover",
        "Demand_Mean",
        "Demand_Std",
        "Lead_Time_Demand",
        "Safety_Stock",
        "Target_Stock",
        "Min_Projected_Inventory",
        "Stockout Risk",
        "Risk Level",
        "Recommended_Order_Qty"
    ]
].copy()

In [30]:
final_risk_analysis.head(20)

,Store ID,Product ID,Current_Inventory,Forecast_7D,Forecast_14D,Avg_Daily_Forecast,Days_of_Cover,Demand_Mean,Demand_Std,Lead_Time_Demand,Safety_Stock,Target_Stock,Min_Projected_Inventory,Stockout Risk,Risk Level,Recommended_Order_Qty
0,S001,P0001,223,798.124883,1596.249767,114.017840,1.955834,137.306430,108.187859,342.053521,309.188333,651.241854,-1373.249767,HIGH,HIGH,429
1,S001,P0002,217,1140.824742,2281.649484,162.974963,1.331493,131.038304,104.313220,488.924889,298.115065,787.039954,-2064.649484,HIGH,HIGH,571
2,S001,P0003,69,1276.393665,2552.787330,182.341952,0.378410,141.440492,107.211225,547.025856,306.397227,853.423083,-2483.787330,HIGH,HIGH,785
3,S001,P0004,338,973.422278,1949.311837,139.236560,2.427523,140.904241,113.644117,417.180976,324.781685,741.962661,-1611.311837,HIGH,HIGH,404
4,S001,P0005,471,1611.750464,3223.500929,230.250066,2.045602,133.129959,107.368873,690.750199,306.847766,997.597965,-2752.500929,HIGH,HIGH,527
5,S001,P0006,305,624.839502,1249.679003,89.262786,3.416877,130.673051,109.015261,267.788358,311.552953,579.341311,-944.679003,HIGH,MEDIUM,275
6,S001,P0007,256,597.856430,1195.712860,85.408061,2.997375,137.823529,111.198898,256.224184,317.793532,574.017716,-939.712860,HIGH,HIGH,319
7,S001,P0008,315,901.743315,1803.486630,128.820474,2.445263,137.132695,108.699045,386.461421,310.649243,697.110664,-1488.486630,HIGH,HIGH,383
8,S001,P0009,167,1454.905189,2909.810379,207.843598,0.803489,131.436389,107.086803,623.530795,306.041642,929.572437,-2742.810379,HIGH,HIGH,763
9,S001,P0010,167,429.742844,859.485688,61.391835,2.720231,132.361149,105.916371,184.175505,302.696684,486.872188,-692.485688,HIGH,HIGH,320


## 21. Sonuç Tablosu ve Dosya Çıktıları

Talep, stok yeterliliği, güvenlik stoğu ve sipariş önerileri tek tabloda birleştirilir.
Sonuçların tamamı Parquet ve CSV biçimlerinde kaydedilir.

Görüntülemede kullanılan head(20), kaydedilen satır sayısını sınırlandırmaz.

In [31]:
OUTPUT_PATH = Path("outputs")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

final_risk_analysis.to_parquet(
    OUTPUT_PATH / "inventory_risk_analysis.parquet",
    index=False
)

print(
    "inventory_risk_analysis.parquet kaydedildi."
)

inventory_risk_analysis.parquet kaydedildi.


In [32]:
final_risk_analysis.to_csv(
    OUTPUT_PATH / "inventory_risk_analysis.csv",
    index=False
)

print("CSV kaydedildi.")

CSV kaydedildi.


## 22.  Kayıtları İnceleme

Risk sınıflarının dağılımı ve toplam sipariş önerisi gösterilir.
En yüksek sipariş miktarları ile stok yeterliliği 3 günden az olan kayıtlar incelenir.

Sipariş miktarı sıralaması ile stok tükenme aciliyeti aynı ölçüt değildir.

In [33]:
print("===== SMARTSUPPLY INVENTORY SUMMARY =====")

print(
    "\nRisk dağılımı:"
)

print(
    final_risk_analysis["Risk Level"]
    .value_counts()
)

print(
    "\nStock-out riski yüksek ürün sayısı:",
    (
        final_risk_analysis["Stockout Risk"] == "HIGH"
    ).sum()
)

print(
    "\nToplam önerilen sipariş:",
    final_risk_analysis["Recommended_Order_Qty"].sum()
)

===== SMARTSUPPLY INVENTORY SUMMARY =====

Risk dağılımı:
HIGH      71
MEDIUM    28
LOW        1
Name: Risk Level, dtype: int64

Stock-out riski yüksek ürün sayısı: 100

Toplam önerilen sipariş: 43198


In [34]:
top_orders = (
    final_risk_analysis
    .sort_values(
        "Recommended_Order_Qty",
        ascending=False
    )
    .head(10)
)

top_orders[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Forecast_14D",
        "Risk Level",
        "Recommended_Order_Qty"
    ]
]

,Store ID,Product ID,Current_Inventory,Forecast_14D,Risk Level,Recommended_Order_Qty
15,S001,P0016,74,3002.018521,HIGH,877
11,S001,P0012,213,3680.698339,HIGH,868
31,S002,P0012,193,3379.479237,HIGH,851
78,S004,P0019,65,2667.836442,HIGH,823
77,S004,P0018,83,2669.754258,HIGH,786
2,S001,P0003,69,2552.787330,HIGH,785
86,S005,P0007,59,2428.114466,HIGH,782
82,S005,P0003,98,2575.589287,HIGH,774
8,S001,P0009,167,2909.810379,HIGH,763
26,S002,P0007,53,2204.805652,HIGH,742


In [35]:
high_risk_products = (
    final_risk_analysis[
        final_risk_analysis["Risk Level"] == "HIGH"
    ]
    .sort_values(
        "Days_of_Cover"
    )
)

high_risk_products[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Days_of_Cover",
        "Forecast_14D",
        "Safety_Stock",
        "Recommended_Order_Qty"
    ]
].head(20)

,Store ID,Product ID,Current_Inventory,Days_of_Cover,Forecast_14D,Safety_Stock,Recommended_Order_Qty
26,S002,P0007,53,0.336538,2204.805652,322.288399,742
86,S005,P0007,59,0.340182,2428.114466,319.731397,782
78,S004,P0019,65,0.341100,2667.836442,316.006965,823
15,S001,P0016,74,0.345101,3002.018521,307.333770,877
2,S001,P0003,69,0.378410,2552.787330,306.397227,785
77,S004,P0018,83,0.435246,2669.754258,296.410459,786
28,S002,P0009,62,0.450190,1928.076828,321.287926,672
82,S005,P0003,98,0.532694,2575.589287,319.870864,774
74,S004,P0015,56,0.710083,1104.095740,309.227556,490
51,S003,P0012,108,0.718795,2103.520111,319.820729,663
